# Optional GeoAI tree extraction with pretrained models
**This notebook runs actual model inference when explicitly enabled.** SamGeo LangSAM combines GroundingDINO detection and Segment Anything segmentation. Start with one small georeferenced **RGB aerial-image tile**, not a canopy/temperature visualization. No training from scratch. Model performance in Plano is unvalidated.

The base project environment does not currently contain SamGeo. Prefer a separate environment or GPU Colab; preserve the existing GIS environment. Installation and weight downloads require internet and may take time.

In [1]:
# Local workaround for inherited incompatible Anaconda PROJ/GDAL paths.
# Affects this notebook kernel only; does not change machine-wide settings.
import os
os.environ.pop('PROJ_DATA', None)
os.environ.pop('GDAL_DATA', None)
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'scripts/environment_pipeline.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/environment_pipeline.py').exists(), 'Open from EmPath or its notebooks folder.'
sys.path.insert(0, str(ROOT / 'scripts'))
import environment_pipeline as ep
config = json.loads((ROOT / 'config/environment_pipeline.json').read_text())
print('Python:', sys.executable)


Python: /Users/nalinikantheti/Documents/ChatGPT/EmPath/.venv/bin/python


## 1 Optional installation
Uncomment only in the environment selected for inference. Restart its kernel after installation if needed. Model licenses/checkpoints and imagery reuse terms must be recorded.

Official workflow: https://samgeo.gishub.org/examples/text_prompts/ ; package installation: https://samgeo.gishub.org/ .

In [2]:
# %pip install 'segment-geospatial[text]'

## 2 Choose imagery and enable inference
Use uint8 RGB bands 1–3, a known CRS and at most 1536 pixels per side. Clip a representative tile using Rasterio first if necessary. Work from authorized imagery downloads such as TxGIO/NAIP; do not scrape Google Maps. Record acquisition date and imagery source. Full four-ZIP tiled inference is future expansion, not included in this small-tile adapter.

In [3]:
RUN_INFERENCE = False
RGB_TILE = None  # e.g. ROOT / 'data/raw/imagery/plano_rgb_tile.tif'
IMAGERY_SOURCE = None
IMAGERY_DATE = None
OUTPUT = ROOT / 'output/geoai/tree_prediction.tif'

In [4]:
if RUN_INFERENCE:
    assert RGB_TILE and Path(RGB_TILE).exists(), 'Set RGB_TILE to a real georeferenced image.'
    assert IMAGERY_SOURCE and IMAGERY_DATE, 'Record source and acquisition date.'
    from geoai_tree_inference import predict_trees
    predict_trees(RGB_TILE, OUTPUT)
    metadata = json.loads(OUTPUT.with_suffix('.json').read_text())
    metadata.update(imagery_source=IMAGERY_SOURCE, source_vintage=IMAGERY_DATE)
    OUTPUT.with_suffix('.json').write_text(json.dumps(metadata, indent=2))
else:
    print('Inference disabled. No AI output generated.')

Inference disabled. No AI output generated.


## 3 Inspect predictions and export polygons
0 = not detected, 1 = predicted tree, 255 = source NoData. These are model predictions, not truth. Compare visible missed trees and false positives on held-out patches. If independent binary reference masks exist, measure IoU/precision/recall; visual inspection alone is not accuracy. Grounding thresholds are model settings, not friction weights.

In [5]:
if RUN_INFERENCE and OUTPUT.exists():
    import rasterio
    import matplotlib.pyplot as plt
    with rasterio.open(OUTPUT) as src:
        plt.imshow(src.read(1, masked=True), vmin=0, vmax=1, cmap='Greens')
        plt.title('Predicted trees — inspect before use')
    ep.polygonize_classes(OUTPUT, [.5], OUTPUT.with_name('tree_classes.gpkg'))
    print('Class 1 contains predicted trees. Class 0 contains non-detections.')
else:
    print('Run inference first to inspect real predictions.')

Run inference first to inspect real predictions.


## 4 Connect to notebook 06
Add a raster spec named `tree_prediction`, path `output/geoai/tree_prediction.tif`, units `binary tree prediction`, valid range 0–1, source/model details and imagery date. Its zonal mean estimates the fraction of valid pixel centers predicted as tree. Local-tile coverage stays explicit; it cannot establish coverage across all four ZIPs.

Keep predicted tree fraction separate from the Texas canopy-percent layer. Neither is automatically time-specific shade. Do not invent missing predictions outside the tile.

Other relevant libraries: [GeoAI](https://opengeoai.org/examples/foundation_models/) for model discovery/adaptation, [TorchGeo](https://torchgeo.readthedocs.io/) for geospatial ML datasets and models, and [exactextract](https://isciences.github.io/exactextract/) for precise raster/polygon summaries (GIS processing, not inference).